<a href="https://colab.research.google.com/github/aniray2908/satellite-esg-risk-engine/blob/main/experiments/python/ceri/ceri_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4 — Corporate Environmental Risk Index (CERI v2)

## Portfolio Expansion & Distribution Stabilization

CERI v2 extends the risk scoring framework from a two-asset prototype
to a four-asset portfolio model.

The objective is to evaluate statistical stability, clustering behavior,
and distributional properties under expanded asset coverage.


## Why Was CERI v2 Necessary?

CERI v1 was constructed using two assets:

- Carajás
- Gevra

While sufficient for validating feature design,
the limited sample size resulted in:

- Binary min-max normalization
- Extreme z-score contrast
- Trivial clustering behavior

To stabilize statistical interpretation and enable meaningful portfolio modeling,
two additional mining assets were added:

- Bingham Canyon (USA)
- Grasberg Mine (Indonesia)

CERI v2 evaluates model behavior across a more diverse exposure distribution.


In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
carajas = pd.read_csv("/content/drive/MyDrive/carajas_pit_centered_full_year_v2_1.csv")
gevra = pd.read_csv("/content/drive/MyDrive/gevra_pit_centered_full_year_v1.csv")
bingham = pd.read_csv("/content/drive/MyDrive/bingham_pit_centered_full_year_v1.csv")
grasberg = pd.read_csv("/content/drive/MyDrive/grasberg_pit_centered_full_year_v1.csv")

carajas["asset"] = "Carajás"
gevra["asset"] = "Gevra"
bingham["asset"] = "Bingham"
grasberg["asset"] = "Grasberg"

df = pd.concat([carajas, gevra, bingham, grasberg], ignore_index=True)

df.head()

,system:index,image_count,low_ndvi_fraction,mean_ndvi,year,.geo,asset
0,0,101,0.865651,0.059416,2019,"{""type"":""MultiPoint"",""coordinates"":[]}",Carajás
1,1,84,0.921274,0.058572,2020,"{""type"":""MultiPoint"",""coordinates"":[]}",Carajás
2,2,89,0.974743,0.012760,2021,"{""type"":""MultiPoint"",""coordinates"":[]}",Carajás
3,3,83,0.954150,0.029671,2022,"{""type"":""MultiPoint"",""coordinates"":[]}",Carajás
4,4,95,0.970843,0.015329,2023,"{""type"":""MultiPoint"",""coordinates"":[]}",Carajás


In [3]:
feature_df = df.groupby("asset").agg({
    "low_ndvi_fraction": ["mean", "var"],
    "mean_ndvi": "mean"
})

feature_df.columns = ["F1_exposure_intensity",
                      "low_ndvi_variance",
                      "mean_ndvi"]

feature_df.reset_index(inplace=True)

# F2: Vegetation Suppression
feature_df["F2_vegetation_suppression"] = 1 - feature_df["mean_ndvi"]

# F3: Persistence (inverse normalized variance)
max_var = feature_df["low_ndvi_variance"].max()
min_var = feature_df["low_ndvi_variance"].min()

feature_df["F3_persistence_raw"] = 1 - (
    (feature_df["low_ndvi_variance"] - min_var) /
    (max_var - min_var + 1e-9)
)

feature_df

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw
0,Bingham,0.999922,2.805335e-09,0.021179,0.978821,1.000000e+00
1,Carajás,0.937332,2.050342e-03,0.035150,0.964850,4.877238e-07
2,Gevra,0.684294,6.057728e-05,0.193744,0.806256,9.704564e-01
3,Grasberg,0.999596,8.161065e-07,-0.011148,1.011148,9.996033e-01


## Feature Summary

F1 — Exposure Intensity  
Mean proportion of exposed land (NDVI < 0.2).

F2 — Vegetation Suppression  
Inverted mean NDVI.

F3 — Persistence  
Inverse of normalized variance in exposure.

With four assets, feature spread is now more informative
and no longer binary.


In [4]:
z_features = ["F1_exposure_intensity",
              "F2_vegetation_suppression",
              "F3_persistence_raw"]

for col in z_features:
    mean_val = feature_df[col].mean()
    std_val = feature_df[col].std()
    feature_df[col + "_z"] = (
        (feature_df[col] - mean_val) /
        (std_val + 1e-9)
    )

feature_df

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw,F1_exposure_intensity_z,F2_vegetation_suppression_z,F3_persistence_raw_z
0,Bingham,0.999922,2.805335e-09,0.021179,0.978821,1.000000e+00,0.629904,0.421693,0.519958
1,Carajás,0.937332,2.050342e-03,0.035150,0.964850,4.877238e-07,0.213303,0.268879,-1.499415
2,Gevra,0.684294,6.057728e-05,0.193744,0.806256,9.704564e-01,-1.470942,-1.465874,0.460299
3,Grasberg,0.999596,8.161065e-07,-0.011148,1.011148,9.996033e-01,0.627735,0.775302,0.519157


In [5]:
w1, w2, w3 = 0.5, 0.3, 0.2

feature_df["CERI_z"] = (
    w1 * feature_df["F1_exposure_intensity_z"] +
    w2 * feature_df["F2_vegetation_suppression_z"] +
    w3 * feature_df["F3_persistence_raw_z"]
)

feature_df.sort_values("CERI_z", ascending=False)

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw,F1_exposure_intensity_z,F2_vegetation_suppression_z,F3_persistence_raw_z,CERI_z
3,Grasberg,0.999596,8.161065e-07,-0.011148,1.011148,9.996033e-01,0.627735,0.775302,0.519157,0.650290
0,Bingham,0.999922,2.805335e-09,0.021179,0.978821,1.000000e+00,0.629904,0.421693,0.519958,0.545452
1,Carajás,0.937332,2.050342e-03,0.035150,0.964850,4.877238e-07,0.213303,0.268879,-1.499415,-0.112568
2,Gevra,0.684294,6.057728e-05,0.193744,0.806256,9.704564e-01,-1.470942,-1.465874,0.460299,-1.083173


## Interpretation of CERI_z (v2)

CERI_z now reflects relative deviation from portfolio mean.

Positive values → above-average environmental exposure  
Negative values → below-average exposure  

Unlike v1, mid-range values should now emerge,
reflecting a more stable distribution.


In [6]:
cluster_features = feature_df[[
    "F1_exposure_intensity_z",
    "F2_vegetation_suppression_z",
    "F3_persistence_raw_z"
]]

kmeans = KMeans(n_clusters=2, random_state=42)
feature_df["risk_cluster"] = kmeans.fit_predict(cluster_features)

feature_df

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw,F1_exposure_intensity_z,F2_vegetation_suppression_z,F3_persistence_raw_z,CERI_z,risk_cluster
0,Bingham,0.999922,2.805335e-09,0.021179,0.978821,1.000000e+00,0.629904,0.421693,0.519958,0.545452,0
1,Carajás,0.937332,2.050342e-03,0.035150,0.964850,4.877238e-07,0.213303,0.268879,-1.499415,-0.112568,0
2,Gevra,0.684294,6.057728e-05,0.193744,0.806256,9.704564e-01,-1.470942,-1.465874,0.460299,-1.083173,1
3,Grasberg,0.999596,8.161065e-07,-0.011148,1.011148,9.996033e-01,0.627735,0.775302,0.519157,0.650290,0


In [7]:
score = silhouette_score(cluster_features, feature_df["risk_cluster"])
score

np.float64(0.37383291076400604)

## Cluster Evaluation

Silhouette score measures separation quality:

- Closer to 1 → well-separated clusters
- Near 0 → overlapping clusters
- Negative → poor clustering

With four assets, clustering becomes non-trivial
and reflects meaningful exposure segmentation.

In [8]:
cluster_means = feature_df.groupby("risk_cluster")["CERI_z"].mean()
high_risk_cluster = cluster_means.idxmax()

feature_df["risk_tier"] = feature_df["risk_cluster"].apply(
    lambda x: "High Risk" if x == high_risk_cluster else "Moderate Risk"
)

feature_df[["asset", "CERI_z", "risk_tier"]]

,asset,CERI_z,risk_tier
0,Bingham,0.545452,High Risk
1,Carajás,-0.112568,High Risk
2,Gevra,-1.083173,Moderate Risk
3,Grasberg,0.650290,High Risk


## CERI v2 — Model Evolution Summary

Compared to v1, CERI v2 demonstrates:

- Stabilized feature distribution
- Non-binary z-score scaling
- Meaningful clustering behavior
- Portfolio-level segmentation
- Improved statistical interpretability

The framework now supports:

- Additional asset integration
- Weight optimization
- Advanced clustering methods
- Multi-signal expansion
